# 第10章 画像処理の基礎

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍の入力番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/10/
- 演習の解答: https://ml.kano.ac/solutions/10/

## 画像データの基礎

### 画像の読み込みと表示

**入力 10.1**　サンプル画像の生成と保存

In [ ]:
import numpy as np
from PIL import Image

# 60×60 のRGB画像を作成（3色の縦ストライプ）
img_array = np.zeros((60, 60, 3),
                     dtype=np.uint8)
img_array[:, :20, :] = [255, 0, 0]  # 左:赤
img_array[:, 20:40, :] = [0, 255, 0]  # 中:緑
img_array[:, 40:, :] = [0, 0, 255]  # 右:青

# NumPy配列からPillow画像を作成して保存
img = Image.fromarray(img_array)
img.save("sample.png")
print("画像を保存しました")

**入力 10.2**　画像の読み込みと基本情報の確認

In [ ]:
from PIL import Image

# 画像の読み込み
img = Image.open("sample.png")

# 画像の基本情報
print(f"フォーマット: {img.format}")
print(f"サイズ: {img.size}")  # (幅, 高さ)
print(f"モード: {img.mode}")  # RGB, L, RGBA

### NumPy配列への変換

**入力 10.3**　画像のNumPy配列への変換

In [ ]:
import numpy as np
from PIL import Image

img = Image.open("sample.png")

# NumPy配列に変換
img_array = np.array(img)

print(f"データ型: {img_array.dtype}")
print(f"形状: {img_array.shape}")
print(f"最小値: {img_array.min()}")
print(f"最大値: {img_array.max()}")

# 左上の画素 (0, 0) の RGB 値
print(f"左上の画素値: {img_array[0, 0]}")

### Matplotlibによる画像の表示

**入力 10.4**　Matplotlibによる画像の表示

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

img = Image.open("sample.png")
img_array = np.array(img)

fig, axs = plt.subplots(1, 2, figsize=(8, 3))

# Pillow画像を直接表示
axs[0].imshow(img)
axs[0].set_title("Pillow画像")
axs[0].axis("off")

# NumPy配列として表示
axs[1].imshow(img_array)
axs[1].set_title("NumPy配列")
axs[1].axis("off")

plt.tight_layout()
plt.show()

### RGB色空間

**入力 10.5**　RGB画像のチャンネル分解と表示

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from urllib.request import urlretrieve

# サンプル画像のダウンロード
url = "https://ml.kano.ac/chapters/data/chikuwa.png"
urlretrieve(url, "chikuwa.png")

# RGBA画像を白背景に合成してRGB画像として読み込む
img_rgba = Image.open("chikuwa.png")
background = Image.new("RGB", img_rgba.size, (255, 255, 255))
background.paste(img_rgba, mask=img_rgba.split()[3])
img_array = np.array(background)
print(f"形状: {img_array.shape}")

# カラー画像とR・G・Bの各チャンネルをグレースケールの明暗として表示
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4))
axs[0].imshow(img_array)
axs[0].set_title("カラー画像")
for i, name in enumerate(["Rチャンネル", "Gチャンネル", "Bチャンネル"]):
    axs[i + 1].imshow(img_array[:, :, i],
                      cmap="gray", vmin=0, vmax=255)
    axs[i + 1].set_title(name)
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

### グレースケール変換

#### 方法1：単純平均法

**入力 10.6**　単純平均法によるグレースケール変換

In [ ]:
import numpy as np
from PIL import Image

img = Image.open("sample.png")
img_array = np.array(img, dtype=np.float64)

# 単純平均法
gray_simple = img_array.mean(axis=2)

print(f"形状: {gray_simple.shape}")
print(f"左上: {gray_simple[0, 0]:.1f}")
print(f"中央: {gray_simple[30, 30]:.1f}")

#### 方法2：加重平均法（NTSC/ITU-R BT.601係数）

**入力 10.7**　加重平均法によるグレースケール変換

In [ ]:
import numpy as np
from PIL import Image

img = Image.open("sample.png")
img_array = np.array(img, dtype=np.float64)

# 加重平均法（ITU-R BT.601）
weights = np.array([0.299, 0.587, 0.114])
gray_weighted = np.dot(img_array, weights)

print(f"形状: {gray_weighted.shape}")
print(f"赤領域: {gray_weighted[30, 10]:.4f}")
print(f"緑領域: {gray_weighted[30, 30]:.4f}")
print(f"青領域: {gray_weighted[30, 50]:.4f}")

#### 方法3：Pillowのconvertメソッド

**入力 10.8**　`convert("L")`によるグレースケール変換

In [ ]:
from PIL import Image
import numpy as np

img = Image.open("sample.png")

# Pillowのconvertメソッドでグレースケール変換
gray_img = img.convert("L")

print(f"モード: {gray_img.mode}")
print(f"サイズ: {gray_img.size}")

gray_array = np.array(gray_img)
print(f"配列の形状: {gray_array.shape}")
print(f"赤領域: {gray_array[30, 10]}")
print(f"緑領域: {gray_array[30, 30]}")
print(f"青領域: {gray_array[30, 50]}")

## しきい値処理

### 固定しきい値処理

**入力 10.9**　固定しきい値による二値化

In [ ]:
import numpy as np
from PIL import Image

# グレースケールのグラデーション画像を作成
gradient = np.tile(np.arange(256, dtype=np.uint8), (100, 1))
img_gray = Image.fromarray(gradient)
img_gray.save("gradient.png")

# NumPy配列に変換
gray_array = np.array(img_gray)

# 固定しきい値 T=128 で二値化
T = 128
binary = np.where(gray_array >= T, 255, 0).astype(np.uint8)

print(f"入力画像の画素値の範囲: {gray_array.min()} - {gray_array.max()}")
print(f"しきい値: {T}")
print(f"二値化後のユニーク値: {np.unique(binary)}")

binary_img = Image.fromarray(binary)
binary_img.save("binary_fixed.png")
print("二値化画像を保存しました")

### 大津の二値化

**入力 10.10**　大津の二値化

In [ ]:
import numpy as np
from PIL import Image
from skimage.filters import threshold_otsu

# テスト用の画像を作成（2つの明るさ領域を持つ画像）
img_array = np.zeros((100, 100), dtype=np.uint8)
img_array[:50, :] = 80    # 上半分: 暗い
img_array[50:, :] = 200   # 下半分: 明るい

# ノイズを追加
rng = np.random.default_rng(42)
noise = rng.normal(0, 20, img_array.shape)
img_noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)

# 大津の二値化でしきい値を自動決定
thresh = threshold_otsu(img_noisy)
print(f"大津の二値化で求めたしきい値: {thresh}")

# 二値化
binary_otsu = (img_noisy >= thresh).astype(np.uint8) * 255
print(f"二値化後のユニーク値: {np.unique(binary_otsu)}")

## フィルタリング

### ガウシアンフィルタ

**入力 10.11**　ガウシアンフィルタによるノイズ除去

In [ ]:
import numpy as np
from PIL import Image
from urllib.request import urlretrieve
from skimage.filters import gaussian

# サンプル画像のダウンロード
url = "https://ml.kano.ac/chapters/data/hanpen.png"
urlretrieve(url, "hanpen.png")

# RGBA画像を白背景に合成してからグレースケール化（0〜1に正規化）
img_rgba = Image.open("hanpen.png")
background = Image.new("RGB", img_rgba.size, (255, 255, 255))
background.paste(img_rgba, mask=img_rgba.split()[3])
img_clean = np.array(background.convert("L")) / 255.0

# ガウスノイズを追加
rng = np.random.default_rng(42)
noise = rng.normal(0, 0.3, img_clean.shape)
img_noisy = np.clip(img_clean + noise, 0, 1)

# ガウシアンフィルタでノイズ除去
img_filtered_s1 = gaussian(img_noisy, sigma=1)
img_filtered_s2 = gaussian(img_noisy, sigma=2)

print(f"ノイズ画像 - 平均: {img_noisy.mean():.4f}", end=", ")
print(f"標準偏差: {img_noisy.std():.4f}")
print(f"sigma=1   - 平均: {img_filtered_s1.mean():.4f}", end=", ")
print(f"標準偏差: {img_filtered_s1.std():.4f}")
print(f"sigma=2   - 平均: {img_filtered_s2.mean():.4f}", end=", ")
print(f"標準偏差: {img_filtered_s2.std():.4f}")

**入力 10.12**　ノイズ除去結果の比較表示

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 4, figsize=(12, 3.4))

axs[0].imshow(img_clean, cmap="gray", vmin=0, vmax=1)
axs[0].set_title("元画像")

axs[1].imshow(img_noisy, cmap="gray", vmin=0, vmax=1)
axs[1].set_title("ノイズあり")

axs[2].imshow(img_filtered_s1, cmap="gray", vmin=0, vmax=1)
axs[2].set_title("ぼかし (sigma=1)")

axs[3].imshow(img_filtered_s2, cmap="gray", vmin=0, vmax=1)
axs[3].set_title("ぼかし (sigma=2)")

for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

## エッジ検出

### Sobelフィルタ

**入力 10.13**　Sobelフィルタによるエッジ検出

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# サンプル画像（skimage 付属のカメラマン画像、グレースケール）
from skimage.data import camera
img = camera()

# Sobelフィルタの適用
sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)  # x方向
sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)  # y方向

# エッジ強度の計算
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_mag = np.uint8(sobel_mag / sobel_mag.max() * 255)

# 結果の表示
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4))
axs[0].imshow(img, cmap="gray")
axs[0].set_title("元画像")
axs[1].imshow(np.abs(sobel_x), cmap="gray")
axs[1].set_title("Sobel X")
axs[2].imshow(np.abs(sobel_y), cmap="gray")
axs[2].set_title("Sobel Y")
axs[3].imshow(sobel_mag, cmap="gray")
axs[3].set_title("エッジ強度")
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Cannyエッジ検出

**入力 10.14**　Cannyエッジ検出の適用

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# サンプル画像（10.3.1項でダウンロードした hanpen.png）
img_rgba = Image.open("hanpen.png")
background = Image.new("RGB", img_rgba.size, (255, 255, 255))
background.paste(img_rgba, mask=img_rgba.split()[3])
img = np.array(background.convert("L"))  # 0〜255 の uint8

# ガウシアン平滑化でノイズを抑えてからCannyエッジ検出を適用
img_blur = cv2.GaussianBlur(img, (5, 5), 0)
edges = cv2.Canny(img_blur, threshold1=50, threshold2=150)

# 結果の表示
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(img, cmap="gray")
axs[0].set_title("元画像")
axs[1].imshow(edges, cmap="gray")
axs[1].set_title("Cannyエッジ")
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

## HOG特徴量

### scikit-imageによるHOGの実装

**入力 10.15**　HOG特徴量の計算と可視化

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.feature import hog

# サンプル画像（入力10.11でダウンロードした hanpen.png）
img_rgba = Image.open("hanpen.png")
background = Image.new("RGB", img_rgba.size, (255, 255, 255))
background.paste(img_rgba, mask=img_rgba.split()[3])
img = np.array(background.convert("L"))

# HOG特徴量の計算（可視化画像も取得）
features, hog_image = hog(
    img,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=True,
    feature_vector=True
)

print(f"HOG特徴量の次元数: {features.shape[0]}")

# 結果の表示
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img, cmap="gray")
axs[0].set_title("元画像")
axs[1].imshow(hog_image, cmap="gray")
axs[1].set_title("HOG可視化")
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 演習問題

### 演習 10-1: グレースケール変換の比較

10.3.1 項でダウンロードするキャラクター「はんぺん」の画像（hanpen.png）を使って、グレースケール変換の違いを確認してください。はんぺんの画像の代わりに、skimage 付属の coffee 画像（`data.coffee()`）でも実施できます。

**タスク**：

1. hanpen.png をダウンロードし、白背景に合成して RGB 画像として読み込み、形状を確認する（10.1.6 項と同じ手順）
2. 単純平均法（`mean(axis=2)`）でグレースケールに変換する
3. 加重平均法（ITU-R BT.601 係数、`np.dot`）でグレースケールに変換する
4. 元のカラー画像、単純平均法、加重平均法の 3 つを 1 行 3 列で並べて表示する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from urllib.request import urlretrieve

# 1. hanpen.png をダウンロードし、白背景に合成して RGB 画像として読み込む

# 2. 単純平均法でグレースケールに変換

# 3. 加重平均法（ITU-R BT.601）でグレースケールに変換

# 4. 元のカラー画像、単純平均法、加重平均法の 3 つを 1 行 3 列で並べて表示

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-1)

### 演習 10-2: ノイズ除去とフィルタ強度

10.3.1 項と同じ手順で「はんぺん」の画像にガウシアンノイズを追加し、`skimage.filters.gaussian` の `sigma` を 3 通り変えてノイズ除去の強度の違いを比較してください。skimage 付属の chelsea 画像（猫のカラー画像）でも実施できます（カラー画像には `channel_axis=-1` を指定します）。

**タスク**：

1. hanpen.png を白背景に合成してグレースケール化し、0〜1 に正規化する（10.3.1 項と同じ手順）
2. ガウシアンノイズを追加する（平均 0、標準偏差 0.3、`np.random.default_rng(42)`）
3. `gaussian` を **sigma=0.5, 2, 5** の 3 通りで適用する
4. ノイズあり画像と 3 通りのフィルタ結果（合計 4 枚）を 1 行 4 列で並べて表示する
5. sigma を大きくしたときに何が起こるか（ノイズの減り方・輪郭のぼやけ方）をコメントで考察する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from urllib.request import urlretrieve
from skimage.filters import gaussian

# 1. hanpen.png を白背景に合成してグレースケール化し、0〜1 に正規化

# 2. ガウシアンノイズを追加（平均 0、標準偏差 0.3、seed=42）

# 3. sigma=0.5, 2, 5 で gaussian を適用

# 4. ノイズあり + フィルタ 3 通りを 1 行 4 列で並べて表示

# 5. 考察（sigma を大きくしたときの影響）をコメントで記述

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-2)

### 演習 10-3: しきい値処理

「はんぺん」の画像をグレースケール（0〜255）で読み込み、固定しきい値と大津の二値化を比較してください。skimage 付属の coins 画像でも実施できます。

**タスク**：

1. hanpen.png を白背景に合成してグレースケールで読み込む（10.4.3 項と同じ手順、0〜255 の uint8）
2. 固定しきい値 **T=64, 128, 192** の 3 通りで二値化し、元画像と並べて表示する
3. しきい値によって抽出される領域がどのように変わるか観察する
4. `skimage.filters.threshold_otsu` で大津の二値化のしきい値を求め、固定しきい値の結果と比較する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_otsu

# 1. hanpen.png を白背景に合成してグレースケールで読み込む（0〜255 の uint8）

# 2. 固定しきい値 T=64, 128, 192 で二値化し、元画像と並べて表示

# 3. しきい値による違いを観察

# 4. 大津の二値化のしきい値を求め、固定しきい値と比較

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-3)

### 演習 10-4: エッジ検出

「はんぺん」の画像に対して、4 種類のエッジ検出手法（Sobel・Prewitt・Scharr・Canny）の結果を比較してください。skimage 付属の camera 画像でも実施できます。

**タスク**：

1. hanpen.png を白背景に合成してグレースケールで読み込む（10.4.3 項と同じ手順）
2. `filters.sobel`・`filters.prewitt`・`filters.scharr`・`cv2.Canny`（しきい値 (50, 150)）の 4 種類を適用し、2×2 で並べて表示する
3. Canny の 2 つのしきい値を **(50, 150), (100, 200), (10, 50)** と変えて適用し、検出されるエッジの量がどのように変わるか比較する

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import filters

# 1. hanpen.png を白背景に合成してグレースケールで読み込む

# 2. Sobel・Prewitt・Scharr・Canny（しきい値 (50, 150)）を適用し、2×2 で表示

# 3. Canny のしきい値を (50, 150), (100, 200), (10, 50) と変えて比較

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-4)

### 演習 10-5: HOG 特徴量とパラメータ

「はんぺん」の画像に対して HOG 特徴量を 3 通りの `pixels_per_cell` で抽出し、可視化と特徴量の次元数を比較してください。skimage 付属の chelsea 画像（グレースケールに変換して使用）でも実施できます。

**タスク**：

1. hanpen.png を白背景に合成してグレースケールで読み込む（10.5.5 項と同じ手順）
2. `pixels_per_cell` を **(8, 8), (16, 16), (32, 32)** の 3 通りで HOG 特徴量を抽出する（`orientations=9`, `cells_per_block=(2, 2)`, `visualize=True`）
3. 元画像と 3 つの HOG 可視化結果を 2×2 で並べて表示する
4. 各設定での特徴量の次元数を出力する
5. セルを大きくすると何が変わるか（次元数・失われる情報）をコメントで考察する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog

# 1. hanpen.png を白背景に合成してグレースケールで読み込む

# 2. pixels_per_cell を (8, 8), (16, 16), (32, 32) と変えて HOG を抽出

# 3. 元画像 + 3 通りの HOG 可視化を 2×2 で表示

# 4. 各設定での特徴量の次元数を出力

# 5. 考察（セルを大きくすると失われる情報）をコメントで記述

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-5)

### 発展課題 1: 文字画像の二値化

scikit-image のサンプル画像（page）は、スキャンしたページの画像で、左側が暗く右側が明るいという明るさのムラがあります。この画像に対して、以下の 3 通りで二値化を行い、結果を比較してください。

- (a) 大津の二値化
- (b) ガウシアンフィルタ + 大津の二値化
- (c) 適応的しきい値処理 [`cv2.adaptiveThreshold`](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)（使い方・パラメータは公式ドキュメントを参照）

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.filters import gaussian, threshold_otsu

# 1. page 画像を読み込む

# 2-a. 大津の二値化のみ（threshold_otsu）

# 2-b. ガウシアンフィルタ（gaussian） + 大津の二値化

# 2-c. 適応的しきい値処理

# 3. 元画像と 3 通りの結果を 2×2 で並べて表示

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-adv-1)

### 発展課題 2: カラー画像の RGB ヒストグラム

scikit-image のサンプル画像（coffee）の RGB 各チャンネルのヒストグラムを描画し、画像の色の傾向を読み取ってください。

**タスク**：

1. coffee 画像を読み込む
2. R・G・B 各チャンネルのヒストグラムを `numpy.histogram` で計算する
3. 元画像と 3 色のヒストグラムを 1 行 2 列で並べて表示する
4. 各チャンネルの **最頻値・中央値・平均値** を計算して出力する
5. ヒストグラムと出力した統計量から coffee 画像の色の傾向を読み取り、コメントで考察する

In [ ]:
from skimage import data
import matplotlib.pyplot as plt
import numpy as np

# 1. coffee 画像を読み込む

# 2. R・G・B 各チャンネルのヒストグラムを計算

# 3. 元画像と RGB ヒストグラムを 1 行 2 列で表示

# 4. 各チャンネルの最頻値・中央値・平均値を計算して出力

# 5. 考察（色の傾向）をコメントで記述

[解答例を見る](https://ml.kano.ac/solutions/10/#solution-10-adv-2)